# Draft Helper

Startup notebook for loading hitter/pitcher data from pybaseball cache and browsing in tabbed grids.

In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path
from typing import Iterable

import pandas as pd
import ipywidgets as widgets
import ipydatagrid as dg
from pybaseball import batting_stats, pitching_stats, cache

cache.enable()
print(cache.config.cache_directory)

In [ ]:
CURRENT_YEAR = date.today().year
YEARS_TO_LOAD = [CURRENT_YEAR - 3, CURRENT_YEAR - 2, CURRENT_YEAR - 1]


def _read_cache_file(path: Path) -> pd.DataFrame | None:
    try:
        if path.suffix == '.csv':
            return pd.read_csv(path)
        if path.suffix in {'.parquet', '.pq'}:
            return pd.read_parquet(path)
        if path.suffix in {'.pkl', '.pickle'}:
            return pd.read_pickle(path)
    except Exception:
        return None
    return None


def _find_cached_frames(cache_dir: Path, keywords: Iterable[str], years: list[int]) -> pd.DataFrame | None:
    if not cache_dir.exists():
        return None

    suffixes = {'.csv', '.parquet', '.pq', '.pkl', '.pickle'}
    year_tokens = {str(y) for y in years}
    candidates = []
    for path in cache_dir.rglob('*'):
        if path.suffix.lower() not in suffixes:
            continue
        stem = path.stem.lower()
        if not any(k in stem for k in keywords):
            continue
        if year_tokens and not any(token in stem for token in year_tokens):
            continue
        candidates.append(path)

    frames: list[pd.DataFrame] = []
    for path in sorted(candidates):
        frame = _read_cache_file(path)
        if frame is not None and not frame.empty:
            frames.append(frame)

    if not frames:
        return None

    combined = pd.concat(frames, ignore_index=True)
    return combined.drop_duplicates().reset_index(drop=True)


def _fallback_pull(years: list[int], source: str) -> pd.DataFrame:
    pulled = []
    for year in years:
        if source == 'hitters':
            frame = batting_stats(year, qual=0)
        else:
            frame = pitching_stats(year, qual=0)
        if 'Season' not in frame.columns:
            frame = frame.copy()
            frame['Season'] = year
        pulled.append(frame)
    return pd.concat(pulled, ignore_index=True)


cache_dir = Path(cache.config.cache_directory)

hitters_raw = _find_cached_frames(
    cache_dir=cache_dir,
    keywords=['batting_stats', 'batting', 'hitters'],
    years=YEARS_TO_LOAD,
)
if hitters_raw is None:
    hitters_raw = _fallback_pull(YEARS_TO_LOAD, source='hitters')

pitchers_raw = _find_cached_frames(
    cache_dir=cache_dir,
    keywords=['pitching_stats', 'pitching', 'pitchers'],
    years=YEARS_TO_LOAD,
)
if pitchers_raw is None:
    pitchers_raw = _fallback_pull(YEARS_TO_LOAD, source='pitchers')

print(f'Loaded hitter rows: {len(hitters_raw):,}')
print(f'Loaded pitcher rows: {len(pitchers_raw):,}')

In [ ]:
def normalize_core_columns(df: pd.DataFrame, player_type: str) -> pd.DataFrame:
    col_aliases = {
        'PlayerName': ['Name', 'name', 'player_name', 'player'],
        'PlayerID': ['IDfg', 'ID', 'playerid', 'player_id', 'mlb_id'],
        'Team': ['Team', 'Tm', 'team'],
        'Pos': ['Pos', 'position', 'Position'],
        'Season': ['Season', 'season', 'year', 'Year'],
    }

    normalized = pd.DataFrame(index=df.index)
    for canonical, aliases in col_aliases.items():
        for alias in aliases:
            if alias in df.columns:
                normalized[canonical] = df[alias]
                break
        if canonical not in normalized.columns:
            normalized[canonical] = pd.NA

    if player_type == 'hitters':
        ranking_stats = ['HR', 'SB', 'R', 'RBI', 'AVG', 'OBP', 'SLG', 'wRC+', 'WAR']
        sort_cols = [c for c in ['WAR', 'HR', 'SB'] if c in df.columns]
    else:
        ranking_stats = ['W', 'SV', 'SO', 'K/9', 'ERA', 'WHIP', 'WAR']
        sort_cols = [c for c in ['WAR', 'SO', 'SV'] if c in df.columns]

    for col in ranking_stats:
        normalized[col] = df[col] if col in df.columns else pd.NA

    if sort_cols:
        normalized = normalized.sort_values(by=sort_cols, ascending=[False] * len(sort_cols), na_position='last')

    normalized = normalized.reset_index(drop=True)
    normalized.insert(0, 'Rank', normalized.index + 1)
    return normalized


hitters = normalize_core_columns(hitters_raw, player_type='hitters')
pitchers = normalize_core_columns(pitchers_raw, player_type='pitchers')

hitters.head(3), pitchers.head(3)

In [ ]:
hitter_grid = dg.DataGrid(
    hitters,
    selection_mode='row',
    base_row_size=28,
    layout=widgets.Layout(width='100%', height='650px'),
)

pitcher_grid = dg.DataGrid(
    pitchers,
    selection_mode='row',
    base_row_size=28,
    layout=widgets.Layout(width='100%', height='650px'),
)

tab = widgets.Tab(children=[hitter_grid, pitcher_grid])
tab.set_title(0, 'Hitters')
tab.set_title(1, 'Pitchers')

display(tab)